In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

In [ ]:
%config InlineBackend.figure_format = 'retina'

g = 9.81

def energy(theta, omega, L, m=1.0):
    return 0.5 * m * (L**2) * (omega**2) + m * g * L * (1 - np.cos(theta))

def theta_analytic_small_angle(theta0, omega0, L, t):
    # предполагаем начальную скорость omega0 = 0 обычно
    w0 = np.sqrt(g / L)
    return theta0 * np.cos(w0 * t) + (omega0 / w0) * np.sin(w0 * t)

Здесь реализуем три интегратора:

1. explicit Euler:
   $$\text{theta}_{n+1} = \text{theta}_n + \text{dt} \cdot \text{omega}_n$$

   $$\text{omega}_{n+1} = \text{omega}_n + \text{dt} \cdot (-\frac{g}{L} \cdot \sin(\text{theta}_n))$$

2. Euler-Cromer (semi-implicit, часто лучше для осцилляторов):
   $$\text{omega}_{n+1} = \text{omega}_n + \text{dt} \cdot (-\frac{g}{L} \cdot \sin(\text{theta}_n))$$

   $$\text{theta}_{n+1} = \text{theta}_n + \text{dt} \cdot \text{omega}_{n+1}$$

3. RK4 для системы (theta, omega) — классический метод Рунге-Кутта 4-го порядка.

In [ ]:
# Блок 1: интеграторы
def step_euler(theta, omega, L, dt):
    theta_new = theta + dt * omega
    omega_new = omega + dt * (-g / L * np.sin(theta))
    return theta_new, omega_new

def step_euler_cromer(theta, omega, L, dt):
    omega_new = omega + dt * (-g / L * np.sin(theta))
    theta_new = theta + dt * omega_new
    return theta_new, omega_new

def step_rk4(theta, omega, L, dt):
    # система y = [theta, omega]; y' = f(y)
    def f(thet, ome):
        return np.array([ome, -g / L * np.sin(thet)])
    y0 = np.array([theta, omega])
    k1 = f(*y0)
    k2 = f(*(y0 + 0.5*dt*k1))
    k3 = f(*(y0 + 0.5*dt*k2))
    k4 = f(*(y0 + dt*k3))
    y_new = y0 + dt*(k1 + 2*k2 + 2*k3 + k4)/6.0
    return float(y_new[0]), float(y_new[1])


Напишем функцию simulate(method, ...) которая запускает интегратор на заданном dt и возвращает массивы времени, theta, omega и энергии.
Потом сравним методы при малом угле (theta0 = 0.1 rad) и покажем:
- theta(t) (числ. vs анал.)
- фазовый портрет
- энергия vs t

In [ ]:

# Блок 2: симулятор
def simulate(step_func, theta0, omega0, L, dt, T):
    n_steps = int(np.ceil(T / dt))
    t = np.linspace(0, n_steps*dt, n_steps+1)
    theta = np.zeros_like(t)
    omega = np.zeros_like(t)
    E = np.zeros_like(t)
    theta[0] = theta0
    omega[0] = omega0
    E[0] = energy(theta0, omega0, L)
    for i in range(n_steps):
        theta[i+1], omega[i+1] = step_func(theta[i], omega[i], L, dt)
        E[i+1] = energy(theta[i+1], omega[i+1], L)
    return t, theta, omega, E

# параметры
L = 1.0
theta0 = 0.1  # малый угол
omega0 = 0.0
T = 20.0
dt = 0.01

# прогон для трёх методов
t_e, th_e, om_e, E_e = simulate(step_euler, theta0, omega0, L, dt, T)
t_ec, th_ec, om_ec, E_ec = simulate(step_euler_cromer, theta0, omega0, L, dt, T)
t_rk, th_rk, om_rk, E_rk = simulate(step_rk4, theta0, omega0, L, dt, T)

# аналитика (малые углы)
th_analytic = theta_analytic_small_angle(theta0, omega0, L, t_rk)

# --- Плот 1: theta(t)
plt.figure(figsize=(10,5))
plt.plot(t_e, th_e, label='Euler (явный)', alpha=0.6)
plt.plot(t_ec, th_ec, label='Euler-Cromer', alpha=0.8)
plt.plot(t_rk, th_rk, label='RK4', alpha=0.9)
plt.plot(t_rk, th_analytic, '--', label='Аналитическое (малые углы)')
plt.xlabel('t (s)')
plt.ylabel('theta (rad)')
plt.legend()
plt.title('theta(t) - сравнение методов')
plt.show()

# --- Плот 2: фазовый портрет
plt.figure(figsize=(6,6))
plt.plot(th_e, om_e, label='Euler', alpha=0.6)
plt.plot(th_ec, om_ec, label='Euler-Cromer', alpha=0.8)
plt.plot(th_rk, om_rk, label='RK4', alpha=0.9)
plt.xlabel('theta')
plt.ylabel('omega')
plt.legend()
plt.title('Фазовый портрет (theta vs omega)')
plt.show()

# --- Плот 3: энергия
plt.figure(figsize=(10,4))
plt.plot(t_e, E_e, label='Euler')
plt.plot(t_ec, E_ec, label='Euler-Cromer')
plt.plot(t_rk, E_rk, label='RK4')
plt.hlines(E_e[0], 0, T, colors='k', linestyles='dotted', label='E0')
plt.xlabel('t (s)')
plt.ylabel('E')
plt.legend()
plt.title('Энергия vs t')
plt.show()


Для малых углов у нас есть аналитическое решение; вычислим ошибку (например, RMS или max) через всё множество шагов до времени T.
Сравним поведение ошибки как функция dt в лог-лог масштабе для методов: Euler (порядок 1), Euler-Cromer (порядок ~1, но более устойчив), RK4 (порядок 4).
Оценим численный порядок методом линейной регрессии на лог-лог.


In [ ]:
# Блок 3: зависимость ошибки от dt
def compute_error_vs_dt(step_func, theta0, omega0, L, T, dt_list):
    errors = []
    for dt in dt_list:
        t, th, om, E = simulate(step_func, theta0, omega0, L, dt, T)
        th_anal = theta_analytic_small_angle(theta0, omega0, L, t)
        err = np.sqrt(np.mean((th - th_anal)**2))  # RMS
        errors.append(err)
    return np.array(errors)

dt_list = np.array([1e-1, 5e-2, 2e-2, 1e-2, 5e-3, 2e-3, 1e-3])
err_e = compute_error_vs_dt(step_euler, theta0, omega0, L, 5.0, dt_list)
err_ec = compute_error_vs_dt(step_euler_cromer, theta0, omega0, L, 5.0, dt_list)
err_rk = compute_error_vs_dt(step_rk4, theta0, omega0, L, 5.0, dt_list)

# лог-лог график
plt.figure(figsize=(7,5))
plt.loglog(dt_list, err_e, 'o-', label='Euler')
plt.loglog(dt_list, err_ec, 's-', label='Euler-Cromer')
plt.loglog(dt_list, err_rk, '^-', label='RK4')
plt.xlabel('dt')
plt.ylabel('RMS error on theta up to T=5s')
plt.legend()
plt.title('Сходимость по dt')
plt.show()

# оценка порядка методом наименьших квадратов (log)
def estimate_order(dt_array, err_array):
    p = np.polyfit(np.log(dt_array), np.log(err_array), 1)
    slope = p[0]
    C = np.exp(p[1])
    return slope, C

print("Оценка порядка (приблизительно):")
print("Euler:", estimate_order(dt_list, err_e)[0])
print("Euler-Cromer:", estimate_order(dt_list, err_ec)[0])
print("RK4:", estimate_order(dt_list, err_rk)[0])


In [ ]:
def theta_analytic_small(theta0, omega0, L, t):
    w0 = np.sqrt(g / L)
    th = theta0 * np.cos(w0 * t) + (omega0 / w0) * np.sin(w0 * t)
    om = -theta0 * w0 * np.sin(w0 * t) + omega0 * np.cos(w0 * t)
    return th, om

# накопительная RMS-ошибка (cumulative RMS up to each time index)
def cumulative_rms(error_array):
    # error_array shape (N,)
    # cumulative RMS up to i: sqrt( mean_{k=0..i} error[k]^2 )
    sq = error_array**2
    csum = np.cumsum(sq)
    n = np.arange(1, len(sq)+1)
    return np.sqrt(csum / n)

In [ ]:
L = 1.0
theta0 = 0.1   # малый угол
omega0 = 0.0
T = 20.0
dt = 0.01

# прогоним методы
t_e, th_e, om_e, E_e = simulate(step_euler, theta0, omega0, L, dt, T)
t_ec, th_ec, om_ec, E_ec = simulate(step_euler_cromer, theta0, omega0, L, dt, T)
t_rk, th_rk, om_rk, E_rk = simulate(step_rk4, theta0, omega0, L, dt, T)

# убеждаемся, что t одинаковые
assert np.allclose(t_e, t_ec) and np.allclose(t_e, t_rk), "Сетки времени должны совпадать"
t = t_e

# аналитическое решение на той же сетке
th_a, om_a = theta_analytic_small(theta0, omega0, L, t)
E_a = 0.5 * (L**2) * om_a**2 + g * L * (1 - np.cos(th_a))

plt.figure(figsize=(10,6))

ax = plt.gca()

# аналитика (пунктир)
ax.plot(th_a, om_a, 'k--', lw=2, label='Аналитика (малые углы)')

# численные траектории (сплошные), с маркерами каждые N точек для видимости направления
methods = {
    'Euler': (th_e, om_e, {'color':'C0'}),
    'Euler-Cromer': (th_ec, om_ec, {'color':'C1'}),
    'RK4': (th_rk, om_rk, {'color':'C2'}),
}
marker_step = max(1, int(len(t)/200))  # не более ~200 маркеров для читаемости
for name, (th_num, om_num, style) in methods.items():
    ax.plot(th_num, om_num, lw=1.2, label=name, **style)
    ax.plot(th_num[::marker_step], om_num[::marker_step], marker='o', linestyle='',
            markersize=4, alpha=0.7, **style)
    arrow_idx = np.arange(0, len(t)-marker_step, marker_step*20)
    for ii in arrow_idx:
        dx = th_num[ii+marker_step] - th_num[ii]
        dy = om_num[ii+marker_step] - om_num[ii]
        ax.arrow(th_num[ii], om_num[ii], dx, dy, head_width=0.002, head_length=0.002,
                 length_includes_head=True, alpha=0.6, color=style.get('color'))

ax.set_xlabel(r'$\theta$ (rad)')
ax.set_ylabel(r'$\omega$ (rad/s)')
ax.set_title('Фазовый портрет: аналитика vs численные методы (малые углы)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

# ---- графики накопительной RMS-ошибки (theta, omega, energy) ----
fig, axs = plt.subplots(3,1, figsize=(8,10), sharex=True)

# ошибки по времени (точечно)
err_theta_e = np.abs(th_e - th_a)
err_theta_ec = np.abs(th_ec - th_a)
err_theta_rk = np.abs(th_rk - th_a)

err_omega_e = np.abs(om_e - om_a)
err_omega_ec = np.abs(om_ec - om_a)
err_omega_rk = np.abs(om_rk - om_a)

err_E_e = np.abs(E_e - E_a)
err_E_ec = np.abs(E_ec - E_a)
err_E_rk = np.abs(E_rk - E_a)

# накопительная RMS
cum_theta_e = cumulative_rms(err_theta_e)
cum_theta_ec = cumulative_rms(err_theta_ec)
cum_theta_rk = cumulative_rms(err_theta_rk)

cum_omega_e = cumulative_rms(err_omega_e)
cum_omega_ec = cumulative_rms(err_omega_ec)
cum_omega_rk = cumulative_rms(err_omega_rk)

cum_E_e = cumulative_rms(err_E_e)
cum_E_ec = cumulative_rms(err_E_ec)
cum_E_rk = cumulative_rms(err_E_rk)

# θ cumulative RMS (лог-ось)
axs[0].semilogy(t, cum_theta_e, label='Euler')
axs[0].semilogy(t, cum_theta_ec, label='Euler-Cromer')
axs[0].semilogy(t, cum_theta_rk, label='RK4')
axs[0].set_ylabel('cumulative RMS error (θ)')
axs[0].legend()
axs[0].grid(True, which='both', linestyle=':')

# ω cumulative RMS
axs[1].semilogy(t, cum_omega_e, label='Euler')
axs[1].semilogy(t, cum_omega_ec, label='Euler-Cromer')
axs[1].semilogy(t, cum_omega_rk, label='RK4')
axs[1].set_ylabel('cumulative RMS error (ω)')
axs[1].legend()
axs[1].grid(True, which='both', linestyle=':')

# E cumulative RMS (лог-ось)
# для энергии иногда удобнее смотреть relative error: cum(|E-Ea|)/E0  — но оставим абсолютную RMS
axs[2].semilogy(t, cum_E_e, label='Euler')
axs[2].semilogy(t, cum_E_ec, label='Euler-Cromer')
axs[2].semilogy(t, cum_E_rk, label='RK4')
axs[2].set_ylabel('cumulative RMS error (E)')
axs[2].set_xlabel('t (s)')
axs[2].legend()
axs[2].grid(True, which='both', linestyle=':')

fig.suptitle('Накопительная RMS-ошибка по координатам и энергии', y=0.95)
plt.tight_layout(rect=[0,0,1,0.96])
plt.show()

Теперь рассмотрим произвольные (включая большие) начальные углы, например theta0 = 2.5 rad/, 
omega0 = 1.0 rad/s.
Нарисуем фазовые портреты (theta в интервале ±π несколько раз) и посмотрим, как ведут себя методы. 
Обрати внимание: явный Euler обычно быстро теряет физику, Euler-Cromer — лучше сохраняет энергию качественно, RK4 дает очень точный результат, но не симплектен — энергия все равно может немного дрейфовать, но гораздо медленнее.


In [ ]:
# Блок 4: большие углы
L = 1.0
theta0 = 2.5
omega0 = 1.0
T = 50.0
dt = 0.01

t_e, th_e, om_e, E_e = simulate(step_euler, theta0, omega0, L, dt, T)
t_ec, th_ec, om_ec, E_ec = simulate(step_euler_cromer, theta0, omega0, L, dt, T)
t_rk, th_rk, om_rk, E_rk = simulate(step_rk4, theta0, omega0, L, dt, T)

plt.figure(figsize=(8,6))
plt.plot(th_e, om_e, label='Euler', alpha=0.6)
plt.plot(th_ec, om_ec, label='Euler-Cromer', alpha=0.8)
plt.plot(th_rk, om_rk, label='RK4', alpha=0.9)
plt.xlim(-np.pi, np.pi)
plt.ylim(-6,6)
plt.xlabel('theta')
plt.ylabel('omega')
plt.legend()
plt.title('Фазовые портреты при больших начальных углах')
plt.show()

plt.figure(figsize=(10,4))
plt.plot(t_e, E_e, label='Euler')
plt.plot(t_ec, E_ec, label='Euler-Cromer')
plt.plot(t_rk, E_rk, label='RK4')
plt.xlabel('t (s)')
plt.ylabel('E')
plt.legend()
plt.title('Энергия при больших углах')
plt.show()


Анимируем движение маятника, рисуя стержень и грузик. В Jupyter используем FuncAnimation и отображение как HTML.


In [ ]:
L = 1.0
theta0 = 1.2
omega0 = 0.0
T = 10.0
dt = 0.02

t, th, om, E = simulate(step_rk4, theta0, omega0, L, dt, T)
x = L * np.sin(th)
y = -L * np.cos(th)

fig, ax = plt.subplots(figsize=(5,5))
ax.set_xlim(-1.2*L, 1.2*L)
ax.set_ylim(-1.2*L, 0.2*L)
line, = ax.plot([], [], lw=2)
point, = ax.plot([], [], 'o', markersize=10)

def init():
    line.set_data([], [])
    point.set_data([], [])
    return line, point

def animate(i):
    thisx = [0.0, float(x[i])]
    thisy = [0.0, float(y[i])]
    line.set_data(thisx, thisy)
    point.set_data([float(x[i])], [float(y[i])])
    return line, point

anim = animation.FuncAnimation(fig, animate, init_func=init,
                               frames=range(len(t)), interval=dt*1000,
                               blit=True)

plt.close(fig)
from IPython.display import HTML
HTML(anim.to_jshtml())


Задача: создать набор из N независимых маятников (не связанных пружинами), у каждого своя длина $L_i$, выбранная так, чтобы его собственная частота была пропорциональна заданным коэффициентам $r_i$:
$$\text{omega}_i = r_i \cdot \text{omega}_\text{ref}$$
Отсюда $L_i = \frac{L_\text{ref}}{r_i^2}$ (т.к. omega ~ $\frac{1}{\sqrt{L}}$).
Мы анимируем их одновременно, строим графики $theta_i(t)$ и их x-положения (можно получить интересные резонансные/периодические картины).


In [ ]:
L_ref = 1.0
r = np.array([1.0, 2.0, 3.0/2.0, 5.0/2.0])  
L_list = L_ref / (r**2)

theta0s = np.array([0.5, 0.5, 0.3, 0.2])
omega0s = np.zeros_like(theta0s)

T = 20.0
dt = 0.01

thetas_list = []
omegas_list = []
t_common = None

for i, Li in enumerate(L_list):
    ti, thi, omi, Ei = simulate(step_rk4, float(theta0s[i]), float(omega0s[i]), float(Li), dt, T)
    if t_common is None:
        t_common = ti.copy()
    if len(ti) != len(t_common):
        print(f"Warning: length mismatch for pendulum {i}: len(ti)={len(ti)} vs len(t_common)={len(t_common)}. Interpolating to common grid.")
        thi_interp = np.interp(t_common, ti, thi)
        omi_interp = np.interp(t_common, ti, omi)
        thetas_list.append(thi_interp)
        omegas_list.append(omi_interp)
    else:
        thetas_list.append(thi)
        omegas_list.append(omi)

thetas = np.vstack(thetas_list)  # shape (n_pend, n_time)
omegas = np.vstack(omegas_list)
t = t_common  

xs = np.array([L_list[i]*np.sin(thetas[i]) for i in range(len(r))])
ys = np.array([ -L_list[i]*np.cos(thetas[i]) for i in range(len(r))])

plt.figure(figsize=(10,5))
for i in range(len(r)):
    plt.plot(t, thetas[i], label=f'pendulum {i+1}, r={r[i]}')
plt.xlabel('t')
plt.ylabel('theta')
plt.legend()
plt.title('theta_i(t) для набора маятников')
plt.show()

k = max(1, int(len(t) / 500))  
frames_idx = np.arange(0, len(t), k)

fig, ax = plt.subplots(figsize=(8,4))
xmin = -0.5
xmax = 0.5 * len(r)
ax.set_xlim(xmin, xmax)
ax.set_ylim(-1.2*L_ref, 0.2)
lines = []
points = []
xx_offsets = np.linspace(0.5, 0.5*len(r), len(r))
for i in range(len(r)):
    line, = ax.plot([], [], lw=2)
    point, = ax.plot([], [], 'o', markersize=8)
    lines.append(line)
    points.append(point)

def init_rows():
    for line, point in zip(lines, points):
        line.set_data([], [])
        point.set_data([], [])
    return lines + points

def animate_rows(frame_i):
    idx = frames_idx[frame_i]
    for j, (line, point) in enumerate(zip(lines, points)):
        x0 = xx_offsets[j]
        xi = [x0, x0 + float(xs[j, idx])]
        yi = [0.0, float(ys[j, idx])]
        line.set_data(xi, yi)
        point.set_data([xi[1]], [yi[1]])
    return lines + points

anim2 = animation.FuncAnimation(fig, animate_rows, init_func=init_rows,
                                frames=len(frames_idx), interval=30, blit=True)
plt.close(fig)
from IPython.display import HTML
HTML(anim2.to_jshtml())
